# Lab 4.1: Training Transformers for Remote Sensing Classification

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates training a Transformer model for remote sensing land cover classification using PyTorch Lightning with distributed training support.

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 3.1 | Data Preprocessing | ✅ Previous |
| Lab 3.2 | Google Earth Engine Acquisition | ✅ Previous |
| Lab 4 | Understanding Transformers | ✅ Previous |
| Lab 4.1 | **Training on Sentinel-2 Data** | 🔄 **Current** |
| Lab 5 | Distributed Training (Multi-GPU) | ⬜ Next |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## Project Context

**Inputs (From Lab 3.1):**
- Sentinel-2 patches (10 spectral bands)
- CORINE labels (12 land cover classes)
- CSV format: `trainSet1_cleaned.csv` and `valSet1_cleaned.csv`

**What You'll Do:**
- Build and train a transformer-based classifier
- Handle multi-GPU training with PyTorch Lightning
- Deploy on HPC with Slurm batch scripts
- Generate model checkpoints for evaluation

**Output:**
- Trained model weights
- Training logs and validation metrics
- Ready for deployment in Lab 6 (evaluation)

---

## Overview

In this lab, you will:
1. **Build a Transformer Model**: Create a transformer architecture for classification
2. **Prepare Data**: Load and preprocess remote sensing data
3. **Train with PyTorch Lightning**: Use Lightning for simplified training loops
4. **Distributed Training**: Scale training across multiple GPUs and nodes
5. **HPC Deployment**: Submit jobs to Slurm-managed HPC systems

## Part 1: Setup and Dependencies

First, import the required libraries.

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np

## Part 2: Define the Transformer Model

We'll use PyTorch Lightning to create a transformer-based classifier. The model uses multi-head self-attention to capture spatial relationships in remote sensing data.

In [ ]:
class TransformerModel(pl.LightningModule):
    """
    Transformer model for land cover classification.
    
    Architecture:
    - Multi-layer transformer encoder-decoder
    - Multi-head self-attention (10 heads)
    - 5 transformer layers
    - 254-dimensional hidden layer
    - 12 output classes
    """
    def __init__(self):
        super(TransformerModel, self).__init__()
        self.validation_step_outputs = []
        self.training_step_outputs = []
        self.automatic_optimization = True
        
        # Hyperparameters
        self.input_dim = 10
        self.num_heads = 10
        self.num_layers = 5
        self.hidden_dim = 254
        self.num_classes = 12
        
        # Define the layers
        self.batch_first = True
        self.transformer = nn.Transformer(
            d_model=self.input_dim,
            nhead=self.num_heads,
            num_encoder_layers=self.num_layers,
            dim_feedforward=self.hidden_dim,
            batch_first=self.batch_first
        )
        self.fc = nn.Linear(self.input_dim, self.num_classes)

    def forward(self, src, tgt):
        """
        Forward pass through the transformer.
        
        Parameters:
        -----------
        src : torch.Tensor
            Source sequence (input features)
        tgt : torch.Tensor
            Target sequence (for transformer decoder)
        
        Returns:
        --------
        torch.Tensor : Classification logits
        """
        out = self.transformer(src, tgt)
        out = self.fc(out)
        return out

    def training_step(self, batch, batch_idx):
        """
        Training step for each batch.
        """
        x, y = batch
        x = x.view(x.size(dim=0), -1, 10)
        x = x.to(torch.float32)
        
        # Convert target to one-hot encoding
        y = F.one_hot(y, self.num_classes)
        
        # Forward pass
        output = self(x, x)
        loss = nn.CrossEntropyLoss()(output, y)
        
        self.training_step_outputs.append(loss)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        """
        Validation step for each batch.
        """
        x, y = batch
        x = x.view(x.size(dim=0), -1, 10)
        x = x.to(torch.float32)
        
        # Convert target to one-hot encoding
        y = F.one_hot(y, self.num_classes)
        
        # Forward pass
        output = self(x, x)
        loss = nn.CrossEntropyLoss()(output, y)
        
        self.validation_step_outputs.append(loss)
        self.log("val_loss", loss)
        return loss

    def configure_optimizers(self):
        """
        Configure the optimizer (Adam with learning rate 1e-2).
        """
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-2)
        return optimizer

## Part 3: Custom Dataset Class

Create a PyTorch Dataset for loading remote sensing data.

In [ ]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for remote sensing data.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Feature data (n_samples, n_features)
    labels : numpy.ndarray
        Target labels (n_samples,)
    """
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

## Part 4: Data Loading and Preprocessing

Load the training and validation datasets from CSV files.

In [ ]:
# Load data from CSV files
# Note: Update paths as needed for your environment
training_data = np.loadtxt("/p/project/training2328/lab4_1/data/trainSet1_cleaned.csv", 
                          delimiter=",", dtype=int)
validation_data = np.loadtxt("/p/project/training2328/lab4_1/data/valSet1_cleaned.csv", 
                            delimiter=",", dtype=int)

In [ ]:
# Extract features and labels
X_train = training_data[:, 1:] * 0.0001  # Scale features
y_train = training_data[:, 0]            # Labels in first column

X_val = validation_data[:, 1:] * 0.0001
y_val = validation_data[:, 0]

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Features per sample: {X_train.shape[1]}")

## Part 5: Initialize Model and Data Loaders

Create the model and data loaders for training.

In [ ]:
# Hyperparameters
batch_size = 512

# Initialize model
model = TransformerModel()

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)

# Create data loaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

## Part 6: Training with PyTorch Lightning

### Single GPU Training

For local development or single GPU training:

In [ ]:
# Train on single GPU (for testing/development)
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=10  # Use fewer epochs for testing
)

# Start training
trainer.fit(model, train_dataloader, val_dataloader)

### Multi-GPU and Multi-Node Training

For distributed training on HPC systems with multiple GPUs and nodes:

In [ ]:
# Get number of GPUs and nodes from Slurm environment
# These environment variables are set by the Slurm job scheduler
num_gpus = int(os.environ.get('SLURM_NTASKS_PER_NODE', 1))
num_nodes = int(os.environ.get('SLURM_JOB_NUM_NODES', 1))

print(f"Training on {num_nodes} nodes with {num_gpus} GPUs per node")

# Set up the trainer with multi-GPU and multi-node support using DDP
trainer = pl.Trainer(
    accelerator="gpu",
    devices=num_gpus,
    strategy="ddp",  # Distributed Data Parallel
    num_nodes=num_nodes,
    max_epochs=150
)

# Train the model with validation
trainer.fit(model, train_dataloader, val_dataloader)

## Part 7: Complete Training Script

Here's the complete training script that can be run as a standalone Python file for HPC submission:

In [ ]:
# This cell shows the complete script structure
# Save as train_transformer.py for HPC submission

"""
if __name__ == '__main__':
    # Load data from CSV files
    training_data = np.loadtxt("/p/project/training2328/lab4_1/data/trainSet1_cleaned.csv", 
                              delimiter=",", dtype=int)
    validation_data = np.loadtxt("/p/project/training2328/lab4_1/data/valSet1_cleaned.csv", 
                                delimiter=",", dtype=int)
    
    # Extract features and labels
    X_train = training_data[:, 1:] * 0.0001
    y_train = training_data[:, 0]
    X_val = validation_data[:, 1:] * 0.0001
    y_val = validation_data[:, 0]
    
    batch_size = 512
    num_gpus = int(os.environ['SLURM_NTASKS_PER_NODE'])
    num_nodes = int(os.environ['SLURM_JOB_NUM_NODES'])

    # Initialize model and datasets
    model = TransformerModel()
    train_dataset = YourCustomDataset(X_train, y_train)
    val_dataset = YourCustomDataset(X_val, y_val)

    # Initialize data loaders
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

    # Set up trainer with multi-GPU and multi-node support
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=num_gpus, 
        strategy="ddp", 
        num_nodes=num_nodes, 
        max_epochs=150
    )

    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)
"""
pass

## Part 8: HPC Deployment with Slurm

### Slurm Submission Script

To run the training on an HPC cluster, use the following Slurm batch script:

In [ ]:
%%bash
# Save this as submit_training.sh
# Submit with: sbatch submit_training.sh

#!/bin/bash
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=4
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node=4
#SBATCH --exclusive
#SBATCH --account=training2328
#SBATCH --output=output.out
#SBATCH --error=error.er
#SBATCH --time=00:20:00
#SBATCH --job-name=JOAOsTorch
#SBATCH --gres=gpu:4 --partition=dc-gpu-devel

module --force purge
module use $OTHERSTAGES 
ml Stages/2023  GCC/11.3.0  OpenMPI/4.1.4
# load virtual environment if needed
source <your_venv>
export CUDA_VISIBLE_DEVICES=0,1,2,3

##### Number of total processes
echo " "
echo " Nodelist       := " $SLURM_JOB_NODELIST
echo " Number of nodes:= " $SLURM_JOB_NUM_NODES
echo " Ntasks per node:= " $SLURM_NTASKS_PER_NODE
echo " Ntasks         := " $SLURM_NTASKS
echo " "

echo ""
echo "Run started at:- "
date
srun --cpu-bind=none python -u train_transformer.py
echo "Run finished at:- "
date

### Submitting the Job

To submit your training job to the HPC cluster:

```bash
# Make sure the script is executable
chmod +x submit_training.sh

# Submit the job
sbatch submit_training.sh

# Monitor job status
squeue -u $USER

# View output
tail -f output.out

# Check for errors
tail -f error.er
```

## Key Configuration Parameters

### Model Architecture
- **Input dimension**: 10 features (spectral bands)
- **Attention heads**: 10 heads for multi-head attention
- **Transformer layers**: 5 encoder-decoder layers
- **Hidden dimension**: 254 for feedforward network
- **Output classes**: 12 land cover types

### Training Configuration
- **Batch size**: 512 samples per batch
- **Learning rate**: 0.01 (Adam optimizer)
- **Epochs**: 150 for full training
- **Loss function**: CrossEntropyLoss

### HPC Configuration
- **Nodes**: 2 compute nodes
- **GPUs per node**: 4 GPUs
- **CPUs per task**: 32 cores
- **Strategy**: Distributed Data Parallel (DDP)
- **Time limit**: 20 minutes (adjust for full training)

## Summary

This notebook demonstrated:

1. **Transformer Architecture**: Built a multi-layer transformer with self-attention for remote sensing classification
2. **PyTorch Lightning**: Simplified training code with automatic optimization and logging
3. **Data Pipeline**: Custom Dataset and DataLoader for efficient data loading
4. **Distributed Training**: DDP strategy for multi-GPU and multi-node training
5. **HPC Integration**: Slurm batch scripts for submitting jobs to HPC clusters

The trained model can classify remote sensing data into 12 land cover categories using transformer-based deep learning.

---

## Next Steps

### Monitoring Your Training

After submitting your job with `sbatch submit_training.sh`:

```bash
# Check job status
squeue -u $USER

# Monitor output in real-time
tail -f output.out

# Once training completes, check outputs
ls -la checkpoints/
```

### Checkpoint Management

PyTorch Lightning saves checkpoints during training:
- Best model based on validation loss: `checkpoints/best_model.ckpt`
- Last model: `checkpoints/last.ckpt`
- Use for evaluation in Lab 6

### Preparing for Lab 5: Distributed Training

If you want to scale to more GPUs/nodes:

1. **Modify Slurm script**:
   ```bash
   #SBATCH --nodes=4          # Increase nodes
   #SBATCH --ntasks-per-node=8 # More GPUs per node
   #SBATCH --gpus-per-node=8
   ```

2. **PyTorch Lightning auto-handles DDP** - no code changes needed!

### Preparing for Lab 6: Validation & Evaluation

Save your best model for the next lab:

```bash
# Copy checkpoint to accessible location
cp checkpoints/best_model.ckpt ~/models/lab4_transformer.ckpt
```

In Lab 6, you'll:
- Load this checkpoint
- Run inference on validation set
- Calculate accuracy metrics (OA, PA, UA)
- Generate confusion matrices
- Compare with other land cover products (WorldCover, Esri)

---

## Troubleshooting

**Q: Training is slow on single GPU**
- Normal! Transformers are computationally intensive
- Move to Lab 5 for distributed training
- Reduce batch size if out of memory

**Q: Job gets killed with "OOM"**
- Reduce `batch_size` in the script
- Reduce number of transformer layers
- Increase time allocation: `#SBATCH --time=01:00:00`

**Q: How long should training take?**
- Single GPU: ~4-6 hours for 150 epochs
- 4 GPUs: ~1-2 hours
- 8 GPUs (Lab 5): ~30-45 minutes

---

## Key Configuration Parameters

### Model Architecture
- **Input dimension**: 10 features (spectral bands)
- **Attention heads**: 10 heads for multi-head attention
- **Transformer layers**: 5 encoder-decoder layers
- **Hidden dimension**: 254 for feedforward network
- **Output classes**: 12 land cover types

### Training Configuration
- **Batch size**: 512 samples per batch
- **Learning rate**: 0.01 (Adam optimizer)
- **Epochs**: 150 for full training
- **Loss function**: CrossEntropyLoss

### HPC Configuration
- **Nodes**: 2 compute nodes
- **GPUs per node**: 4 GPUs
- **CPUs per task**: 32 cores
- **Strategy**: Distributed Data Parallel (DDP)
- **Time limit**: 20 minutes (adjust for full training)

---

**Continue to Lab 5 for distributed training →**